In [1]:
import copy
import h5py
import matplotlib
from matplotlib import pyplot as plt

import nibabel as nib
 
from nilearn.plotting import show
from nilearn import surface, datasets, plotting
import numpy as np

import os
from os.path import join as ospj

#### Load scalars (fsaverage)

In [49]:

def load_scalar(subject_id, hemi, scalar_name, depth, indir):
    filename = f"{subject_id}_{scalar_name}_{hemi}_{depth}_fsaverage.shape.gii"
    file_path = ospj(indir, scalar_name, filename)

    try:
        gii_array = nib.load(file_path)
        scalar_data = gii_array.darrays[0].data.astype(np.float32)
        return scalar_data
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return None


 

In [50]:

subject_id = "sub-2987185"
indir = ospj("/Users/audluo/PennLINC/luowm_local/output/superficial_wm_testing/vol_to_surf_scalars", subject_id, "fsaverage")


# depths tag
depths = ["depth_0", "depth_0p5", "depth_0p75", "depth_1", "depth_1p25", "depth_1p5", "depth_1p75", "depth_2", "depth_2p25", "depth_2p5", "depth_2p75", "depth_3"]  


# Reload the scalar data into the dictionary structure
scalars_vol2surf = {}

for scalar_name in ["ad", "dti_fa", "gfa", "ha", "iso", "md", "qa", "rd", "rd1", "rd2"]:
    scalars_vol2surf[scalar_name] = {}
    
    for hemi in ["lh", "rh"]:
        scalars_vol2surf[scalar_name][hemi] = {}
        
        for depth in depths:
            scalar_data = load_scalar(subject_id, hemi, scalar_name, depth, indir)
            scalars_vol2surf[scalar_name][hemi][depth] = scalar_data

 

#### Load GM probsegs (fsaverage)

In [48]:
probseg_dir = ospj("/Users/audluo/PennLINC/luowm_local/output/superficial_wm_testing/GMprobseg", subject_id, "fsaverage")
gii_files = [file for file in os.listdir(probseg_dir) if file.endswith(".shape.gii")]


GM_probseg = {"lh": {}, "rh": {}}

for file in gii_files:

    parts = os.path.splitext(file)[0].split("_")
    hemi = os.path.splitext(parts[0])[0].split(".")[0]
    depth = "depth_" + parts[2]
    
    # Load the GIfTI file
    file_path = os.path.join(probseg_dir, file)
    data = nib.load(file_path).darrays[0].data.astype(float)

    # Store the data in the GM_probseg
    GM_probseg[hemi][depth] = data

 

#### Assign NA to DSIstudio scalar vertices where GM probseg > 50% -- edit: may need to do this step in R with modelarray?

In [70]:
# Create an empty dictionary with the same structure as scalars_vol2surf
scalars_v2s_GMprob50NA = copy.deepcopy(scalars_vol2surf)

# the various depths
depths = ["depth_0", "depth_0p5", "depth_0p75", "depth_1", "depth_1p25", "depth_1p5", "depth_1p75", "depth_2", "depth_2p25", "depth_2p5", "depth_2p75", "depth_3"]   

for scalar_name in ["ad", "dti_fa", "gfa", "ha", "iso", "md", "qa", "rd", "rd1", "rd2"]:
    for hemi in ["lh", "rh"]:
        for depth in depths:
            # Get the scalar data
            scalar_data = scalars_vol2surf[scalar_name][hemi][depth]

            # Get the corresponding GM probability values
            prob_values = GM_probseg[hemi][depth]

            # Replace vertices in scalar_data with "NA" if corresponding GM probability at that vertex is > 50%
            probseg_scalar_data = np.where(prob_values > 0.5, np.nan, scalar_data)

            # Update the scalar data in scalars_v2s_GMprob50NA
            scalars_v2s_GMprob50NA[scalar_name][hemi][depth] = probseg_scalar_data

             

In [73]:
# save out as numpy arrays and as HDF5 files

# save out scalar numpy arrays (with NAs at appropriate vertices)  
if not os.path.exists(ospj("/Users/audluo/PennLINC/luowm_local/output/superficial_wm_testing/scalars_v2s_GMprob50NA_fsavg/", subject_id)):
        os.makedirs(ospj("/Users/audluo/PennLINC/luowm_local/output/superficial_wm_testing/scalars_v2s_GMprob50NA_fsavg/", subject_id))
        print(f"Directory 'scalars_v2s_GMprob50NA_fsavg' created.")
else:
        print(f"Directory 'scalars_v2s_GMprob50NA_fsavg' already exists.")



output_dir = ospj("/Users/audluo/PennLINC/luowm_local/output/superficial_wm_testing/scalars_v2s_GMprob50NA_fsavg/", subject_id)

# Iterate through the dictionary and save each array
for scalar_name, hemi_data in scalars_v2s_GMprob50NA.items():
    # makes scalar directory in outdir
    final_outdir = ospj(output_dir, scalar_name)
    if not os.path.exists(final_outdir):
        os.makedirs(final_outdir)
        print(f"Directory '{final_outdir}' created.")
    else:
        print(f"Directory '{final_outdir}' already exists.")


    for hemi, depth_data in hemi_data.items():
        for depth, data_array in depth_data.items():
            # Convert unicode strings to bytes to be able to save as HDF5
            #data_array_bytes = np.array(data_array, dtype='S')

            # make filename
            filename = f"{subject_id}_{scalar_name}_{hemi}_{depth}"
            np_output_path = os.path.join(final_outdir, f"{filename}.npy")
            h5_output_path = os.path.join(final_outdir, f"{filename}.h5")

            # Save numpy array
            np.save(np_output_path, data_array)
            print(f"Saved as NumPy file: {np_output_path}")

            # Save HDF5 file
            with h5py.File(h5_output_path, "w") as h5file:
                h5file.create_dataset("data", data=data_array)
            print(f"Saved as HDF5 file: {h5_output_path}")


Directory 'scalars_v2s_GMprob50NA_fsavg' created.
Directory '/Users/audluo/PennLINC/luowm_local/output/superficial_wm_testing/scalars_v2s_GMprob50NA_fsavg/sub-2987185/ad' created.
Saved as NumPy file: /Users/audluo/PennLINC/luowm_local/output/superficial_wm_testing/scalars_v2s_GMprob50NA_fsavg/sub-2987185/ad/sub-2987185_ad_lh_depth_0.npy
Saved as HDF5 file: /Users/audluo/PennLINC/luowm_local/output/superficial_wm_testing/scalars_v2s_GMprob50NA_fsavg/sub-2987185/ad/sub-2987185_ad_lh_depth_0.h5
Saved as NumPy file: /Users/audluo/PennLINC/luowm_local/output/superficial_wm_testing/scalars_v2s_GMprob50NA_fsavg/sub-2987185/ad/sub-2987185_ad_lh_depth_0p5.npy
Saved as HDF5 file: /Users/audluo/PennLINC/luowm_local/output/superficial_wm_testing/scalars_v2s_GMprob50NA_fsavg/sub-2987185/ad/sub-2987185_ad_lh_depth_0p5.h5
Saved as NumPy file: /Users/audluo/PennLINC/luowm_local/output/superficial_wm_testing/scalars_v2s_GMprob50NA_fsavg/sub-2987185/ad/sub-2987185_ad_lh_depth_0p75.npy
Saved as HDF5 fil